# Lesson 26 Lab — Persistent Scheduling and the TMA Boundary

**Puzzle:** When fixed resident programs, tile loops, launch geometry, and architecture features change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates fixed resident programs, tile loops, launch geometry, and architecture features and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A persistent kernel launches fewer programs and lets each program loop over multiple tiles. This can amortize scheduling or support producer-consumer pipelines, but it may reduce flexibility and fairness. TMA is a separate architecture-specific data-movement mechanism, not a synonym for persistence.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["fixed resident programs, tile loops, launch geometry, and architecture features"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Labeling a tile loop 'TMA optimized' without a descriptor, supported load/store path, or profiler evidence is overclaiming.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 26
LESSON_TITLE = 'Persistent Scheduling and the TMA Boundary'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260839
}


## 5. Freeze the experiment

**Experiment:** Compare a 256-program persistent affine grid with the standard one-program-per-tile grid; do not use TMA.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.021631999872624874,
  "secondary": 0.020192000083625317,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "persistent_programs": 256,
    "standard_programs": 16384,
    "persistent_samples_ms": [
      0.028831999748945236,
      0.025567999109625816,
      0.021856000646948814,
      0.0226879995316267,
      0.020927999168634415,
      0.022272000089287758,
      0.020864000543951988,
      0.02252800017595291,
      0.021503999829292297,
      0.022655999287962914,
      0.020447999238967896,
      0.021376000717282295,
      0.02127999998629093,
      0.02175999991595745,
      0.020479999482631683,
      0.02054399996995926,
      0.0208320003002882,
      0.020416000857949257,
      0.024607999250292778,
      0.021824000403285027
    ],
    "standard_samples_ms": [
      0.0272000003606081,
      0.02163200080394745,
      0.021183999255299568,
      0.020128000527620316,
      0.021695999428629875,
      0.02191999927163124,
      

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Persistent median | 0.0216 ms |
| Standard-grid median | 0.0202 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

Persistent scheduling reduced the grid from 16,384 to 256 programs and took 0.0216 ms versus 0.0202 ms for the standard grid. TMA was not used.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Adopt persistence only when its scheduling benefit survives representative concurrency and shape tests.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 26,
  "title": "Persistent Scheduling and the TMA Boundary",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260839
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.021631999872624874,
    "secondary": 0.020192000083625317,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "persistent_programs": 256,
      "standard_programs": 16384,
      "persistent_samples_ms": [
        0.028831999748945236,
        0.025567999109625816,
        0.021856000646948814,
        0.0226879995316267,
        0.020927999168634415,
        0.022272000089287758,
        0.020864000543951988,
        0.02252800017595291,
        0.021503999829292297,
        0.022655999287962914,
        0.0204479992389

## 10. Make the bounded decision

> Adopt persistence only when its scheduling benefit survives representative concurrency and shape tests.

**Failure analysis:** Labeling a tile loop 'TMA optimized' without a descriptor, supported load/store path, or profiler evidence is overclaiming.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
